<a href="https://colab.research.google.com/github/lawcarp5-bit/steam-game-value-finder/blob/main/Module06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### *[MIST 4600] Fall 2026 · Weeks 12-13*
# **Module 6: LLM APIs and GitHub**

* Due: **Sunday, November 15, 11:59 pm** on eLC


## Your last lab notebook
This one file covers this module's two lab days, and it builds your **GitHub project**:

* **Week 12: Using LLM APIs** *(in class, Thursday Nov 5; graded)*: you use Gemini to label Steam games, a tool you can use in your project.
* **Week 13: Your Project on GitHub** *(in class, Thursday Nov 12, after Code Check 5; graded)*: step-by-step instructions to put your project on GitHub: your repository, project notebook and data, README draft, first function, and commits.
* **AI Use Disclosure** *(at the end, required)*

**AI policy:** this lab is **AI-Allowed with disclosure**. Use AI tools if you wish, but **try it yourself first**, then use AI tools if you can't figure it out. You must understand every line you submit, and you must fill in the disclosure at the end (required whether or not you used AI). On the final exam you explain your own project code without AI. Never share your notebook file with another student, and never copy from someone else's. Keep this notebook, with its revision history, in your Google Drive until final grades post.

**⚠️ First step, as in Module 5.**

1. Download **this notebook** from eLC (Module 6).
2. Drag it into your **MIST 4600** folder in [Google Drive](https://drive.google.com). `steam_games.csv` is already there from Module 5; if not, download it from eLC: **GitHub Project → project_data**.
3. **Rename the notebook** to `Module06_FirstLast.ipynb`, using your own name: Jordan Smith would use `Module06_JordanSmith.ipynb`. No spaces, keep the `.ipynb` on the end.
4. Right-click the notebook → **Open with → Google Colaboratory**.

**You also need a Gemini key** in Colab Secrets, named `GOOGLE_API_KEY`, with **Notebook access** turned on for this notebook. Get it at [aistudio.google.com](https://aistudio.google.com) with a **personal Gmail** account: AI Studio is turned off for UGA accounts. (Week 12 slides: "Set Up Your Key".)

---
## How this lab is graded

Grader cells work exactly as in Modules 1 to 5: run the one below each exercise, a **✅** means you got it, an **❌** tells you what to look at. Fix and re-run.

⚠️ **Never edit the grader cells.** They are how your lab is graded.

* **13 items:** the 6 Week 12 exercises, the 6 project steps in Week 13, and the **AI Use Disclosure**.
* Every item is worth the same, and each is **all or nothing**. Exercises and project steps 1-5 are graded by their ✅. Project step 6 and the disclosure are graded on being complete and specific.
* You get **unlimited attempts** before the deadline, so an ❌ is only permanent if you leave it that way.
* **Not graded at all:** the "Finished early?" section.

**New this module:** Gemini's answers change from run to run, so your labels may not match anyone else's exactly. The graders check that your code works, not one exact answer. The project graders look at your **public GitHub repository**, so they need it to exist and be public.

---
# Setup: your data and the libraries

Run the cell below. Colab asks for permission to see your Drive: click **Connect to Google Drive**, choose your account, and allow access. It finds `steam_games.csv` (as in Module 5) and loads it into `games`.

> **You should see:** `Mounted at /content/drive`, then `✅ Loaded 1000 games from: /content/drive/MyDrive/...`. If you see ❌, `steam_games.csv` isn't in your Drive: put it next to this notebook and run the cell again.

In [15]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/


---
# Week 12: Using LLM APIs

*(In class, Thursday Nov 5)*

A publisher is deciding what kind of game to make next and asks: **do multiplayer games do better than single-player games?** Steam's `genres` column doesn't say whether a game is single-player or multiplayer, but each game's `short_description` usually does. Reading 1,000 descriptions by hand would take all day, so you'll have Gemini read them.

Six exercises, in the order of Tuesday's slides: connect, clean the answer, build the prompt, put it in a function, run it on every row, and compare.

## Exercise 1: Connect to Gemini

Connect to Gemini as on the "Connect to Gemini" slide:

1. Import `genai` and `userdata`, read your key from Colab Secrets with `userdata.get("GOOGLE_API_KEY")`, and make `client`.
2. Set `MODEL = "gemini-flash-lite-latest"`.
3. Ask Gemini to explain what Steam is, in one sentence, and store the answer **text** (a string) in `hello`. Print it.

> **You should see:** one sentence about Steam. Yours will be worded differently from your neighbor's.

In [16]:
from google import genai
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)
MODEL = "gemini-flash-lite-latest"

response = client.models.generate_content(
    model=MODEL,
    contents="Explain what Steam is in one sentence."
)
hello = response.text
print(hello)

Steam is an all-in-one digital distribution platform developed by Valve that allows users to buy, play, and update PC games, as well as connect with a global community of gamers.


In [17]:
# ── 🧪 GRADER: Exercise 1  (DO NOT EDIT) ────────────────────────────────────────────────────────────────
try:
    assert MODEL == "gemini-flash-lite-latest", f"MODEL is {MODEL!r}. Set it to \"gemini-flash-lite-latest\"."
    assert isinstance(hello, str), f"hello is a {type(hello).__name__}, not a string. Store response.text, not response."
    assert len(hello.strip()) > 10, "hello is (almost) empty. Print response.text to see what came back."
    print("✅ Exercise 1 passed: you're connected to Gemini")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ The Gemini call failed ({type(e).__name__}: {str(e)[:120]}). 503 or 429 means Google is busy: wait a minute and run it again. Anything about the key: check Colab Secrets (key icon) and Notebook access.")

✅ Exercise 1 passed: you're connected to Gemini


## Exercise 2: Clean up the answer

Gemini doesn't always answer with exactly the word you asked for. It may send `"Multiplayer."`, `" single-player\n"`, or `"It is both"`. Before using an answer, your code has to clean it and check it.

Write a function `clean_label(text, allowed)` that:

1. Takes the answer `text` and a list of `allowed` labels.
2. Removes spaces and line breaks at both ends (`.strip()`), makes it lowercase (`.lower()`), and removes a period at the end (`.strip(".")`).
3. Returns the cleaned text **if** it is in `allowed`, and otherwise returns `"unknown"`.

This is the check from the "Check the Answer" slide, in a function of its own, so you can reuse it in your project.

> **You should see:** `multiplayer`, `single-player`, `unknown`

In [18]:
MODES = ["single-player", "multiplayer", "both"]

def clean_label(text, allowed):
    cleaned = text.strip().lower().strip(".")
    if cleaned in allowed:
        return cleaned
    else:
        return "unknown"
print(clean_label("Multiplayer.", MODES))
print(clean_label(" single-player\n", MODES))
print(clean_label("It is both", MODES))

multiplayer
single-player
unknown


In [19]:
# ── 🧪 GRADER: Exercise 2  (DO NOT EDIT) ────────────────────────────────────────────────────────────────
try:
    tests = [("Multiplayer.", MODES, "multiplayer"), (" single-player\n", MODES, "single-player"), ("BOTH", MODES, "both"),
             ("It is both", MODES, "unknown"), ("  Positive.  ", ["positive", "negative"], "positive"), ("maybe", ["yes", "no"], "unknown")]
    for text, allowed, expected in tests:
        got = clean_label(text, allowed)
        assert got == expected, f"clean_label({text!r}, {allowed}) gave {got!r}, expected {expected!r}. strip, lower, strip(\".\"), then check it is in allowed."
    print("✅ Exercise 2 passed: messy answers in, clean labels out")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")

✅ Exercise 2 passed: messy answers in, clean labels out


## Exercise 3: Build the prompt

Write a function `make_prompt(description)` that **returns** (not prints) a prompt string, on one line, that:

* asks whether this Steam game is mainly **single-player, multiplayer, or both** (all three words, spelled like that),
* asks for **one word only**, and
* puts the game's `description` at the end, after `Description:`.

Use an f-string, as on the "Turn the Prompt into a Function" slide.

> **You should see:** your prompt with Terraria's description at the end

In [20]:
def make_prompt(description):
    return f"Is this Steam game mainly single-player, multiplayer, or both? One word only. Description: {description}"

print(make_prompt(games["short_description"][6]))

Is this Steam game mainly single-player, multiplayer, or both? One word only. Description: Dig, fight, explore, build! Nothing is impossible in this action-packed adventure game. Four Pack also available!


In [21]:
# ── 🧪 GRADER: Exercise 3  (DO NOT EDIT) ────────────────────────────────────────────────────────────────
try:
    p = make_prompt("A cozy farm game about growing crops.")
    assert isinstance(p, str), f"make_prompt returned {type(p).__name__}. Use return, not print."
    assert "A cozy farm game about growing crops." in p, "The description should be inside the prompt. Use an f-string with {description}."
    for word in MODES:
        assert word in p.lower(), f"The prompt should list the choice {word!r}, spelled like that."
    assert "one word" in p.lower(), "Ask for one word only."
    assert p.rstrip().endswith("A cozy farm game about growing crops."), "Put the description at the end of the prompt."
    print("✅ Exercise 3 passed: one prompt, any game")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")

✅ Exercise 3 passed: one prompt, any game


## Exercise 4: Ask Gemini, inside a function

Write a function `get_mode(description)` that puts Exercises 2 and 3 together:

1. Waits 4 seconds with `time.sleep(4)`, to stay under the free-tier limit.
2. Builds the prompt with `make_prompt(description)`.
3. Sends it to Gemini with `client.models.generate_content(...)`.
4. Returns the answer cleaned with `clean_label(..., MODES)`.

> **You should see:** two labels from `MODES`. Terraria is often labeled `both`; yours may differ.

In [29]:
def get_mode(description):
    time.sleep(4)
    prompt = make_prompt(description)
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return clean_label(response.text, MODES)

print(get_mode(games["short_description"][6]))
print(get_mode("Explore a lonely island alone and uncover its story. No online features."))

both
single-player


In [23]:
# ── 🧪 GRADER: Exercise 4  (DO NOT EDIT) ────────────────────────────────────────────────────────────────
try:
    got = get_mode("Play alone through a long story campaign. There is no online or co-op mode.")
    assert isinstance(got, str), f"get_mode returned {type(got).__name__}. Return clean_label(...), don't print it."
    assert got in MODES, f"get_mode gave {got!r} for a clearly single-player game. It should return one of {MODES}. Print response.text inside your function to see what Gemini said."
    print(f"✅ Exercise 4 passed: Gemini said {got!r}, and your function cleaned and checked it")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ The Gemini call failed ({type(e).__name__}: {str(e)[:120]}). 503 or 429 means Google is busy: wait a minute and run it again. Anything about the key: check Colab Secrets (key icon) and Notebook access.")

✅ Exercise 4 passed: Gemini said 'single-player', and your function cleaned and checked it


## Exercise 5: Label 20 games

Now label the 20 games with the most reviews:

1. Make `sample`: the first 20 rows of `games`, with `.head(20).copy()`.
2. Add a column `mode` to `sample` with `.apply(get_mode)` on its `short_description` column. This sends 20 requests and takes about 2 minutes.
3. Print each game's `name` and `mode`.

> **You should see:** 20 games with a label each. Mostly multiplayer games like Counter-Strike 2 and Dota 2 near the top, and story games like The Witcher 3 and ELDEN RING further down.

In [34]:
sample = games.head(20).copy()
sample["mode"] = sample["short_description"].apply(get_mode)

print(sample[["name", "mode"]])

                               name           mode
0                  Counter-Strike 2    multiplayer
1               PUBG: BATTLEGROUNDS           both
2                            Dota 2    multiplayer
3         Grand Theft Auto V Legacy           both
4   Tom Clancy's Rainbow Six® Siege    multiplayer
5                   Team Fortress 2    multiplayer
6                          Terraria           both
7                              Rust    multiplayer
8                       Garry's Mod           both
9                     Apex Legends™    multiplayer
10               Black Myth: Wukong  single-player
11         The Witcher 3: Wild Hunt  single-player
12                       ELDEN RING           both
13                   Stardew Valley           both
14                   Cyberpunk 2077  single-player
15                    HELLDIVERS™ 2           both
16                  Baldur's Gate 3           both
17           Euro Truck Simulator 2           both
18            Red Dead Redempti

In [35]:
# ── 🧪 GRADER: Exercise 5  (DO NOT EDIT) ────────────────────────────────────────────────────────────────
try:
    assert isinstance(sample, pd.DataFrame) and len(sample) == 20, "sample should be games.head(20).copy(): 20 rows."
    assert list(sample["name"]) == list(games["name"].head(20)), "sample should be the first 20 games."
    assert "mode" in sample.columns, "Add the column: sample[\"mode\"] = sample[\"short_description\"].apply(get_mode)"
    bad = set(sample["mode"]) - set(MODES) - {"unknown"}
    assert not bad, f"These labels aren't allowed: {bad}. get_mode should return clean_label(...)."
    n_unknown = (sample["mode"] == "unknown").sum()
    assert n_unknown <= 4, f"{n_unknown} games are 'unknown'. Check your prompt asks for one word only, then run the cell again."
    print(f"✅ Exercise 5 passed: 20 games labeled, {n_unknown} unknown")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")

✅ Exercise 5 passed: 20 games labeled, 0 unknown


## Exercise 6: Do multiplayer games do better?

Back to the publisher's question. Using `sample`:

1. Count how many games got each label with `.value_counts()` on `mode`, and store it in `mode_counts`.
2. Store the average `pct_pos_total` (the percent of positive reviews) for each label in `mode_rating`, with `groupby`, rounded to 1 decimal with `.round(1)`.
3. Print both.

Then, in one sentence below the code, answer the publisher. (Only 20 games, so it's a first look, not a final answer.)

> **You should see:** the counts and the average rating for each label. Your numbers depend on your labels.

In [36]:
mode_counts = sample["mode"].value_counts()
mode_rating = sample.groupby("mode")["pct_pos_total"].mean().round(1)

print(mode_counts)
print(mode_rating)

mode
both             11
multiplayer       6
single-player     3
Name: count, dtype: int64
mode
both             88.1
multiplayer      82.3
single-player    92.3
Name: pct_pos_total, dtype: float64


In [37]:
# ── 🧪 GRADER: Exercise 6  (DO NOT EDIT) ────────────────────────────────────────────────────────────────
try:
    assert mode_counts.equals(sample["mode"].value_counts()), "mode_counts should be sample[\"mode\"].value_counts()."
    expected = sample.groupby("mode")["pct_pos_total"].mean().round(1)
    assert isinstance(mode_rating, pd.Series) and mode_rating.sort_index().equals(expected.sort_index()), "mode_rating should be sample.groupby(\"mode\")[\"pct_pos_total\"].mean().round(1)."
    print("✅ Exercise 6 passed: LLM labels in, a business answer out")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")

✅ Exercise 6 passed: LLM labels in, a business answer out


**My answer to the publisher (one sentence):** *(double-click to edit)*

---
## Finished early? Going further (optional, ungraded)

**A. What do critics say?** The `critic_reviews` column has quotes from critics (some games have none). Write a prompt that summarizes Baldur's Gate 3's critic quotes in one sentence. Then try the same for a game you like.

**B. More games.** Label the top 100 games instead of 20 (about 7 minutes). Does your answer to the publisher change?

---
# Week 13: Your Project on GitHub

*(In class, Thursday Nov 12, after Code Check 5. Finish by Sunday, Nov 15.)*

In this part you put your project on GitHub, one step at a time. You'll do everything here from the beginning, so you don't need to have done anything before; if you already did some of it in Tuesday's lecture, check that it matches each step and move on.

By the end you will have:

* a **public repository** (repo): your project's folder on GitHub,
* your **project notebook** and data in it,
* a **README**: the page visitors see first,
* your **first function** for one of your sub-questions, and
* at least **5 commits**: each time you save something to GitHub, it records a snapshot called a commit, with a short message saying what changed.

You need your **proposal** (your app's one sentence and your sub-questions) next to you.

**Run the cell below first.** It holds the code the graders use to look at your repository on GitHub. You don't need to understand it; just run it.

In [38]:
# ── 🔧 GitHub checker for the graders below (DO NOT EDIT) ────────────────────────
import re
import requests

_GH_CACHE = {}

def _repo_parts(url):
    m = re.search(r"github\.com/([^/\s]+)/([^/\s#?]+)", str(url))
    if not m:
        return None
    return m.group(1), m.group(2).removesuffix(".git")

def _gh(path):
    """GET from GitHub's public API, remembered so each grader doesn't ask again."""
    if path not in _GH_CACHE:
        r = requests.get("https://api.github.com" + path, headers={"Accept": "application/vnd.github+json"}, timeout=20)
        if r.status_code in (403, 429) and "rate limit" in r.text.lower():
            raise RuntimeError("GitHub's limit for checks is used up for now. Wait about an hour and run this grader again.")
        _GH_CACHE[path] = r
    return _GH_CACHE[path]

def _my_repo():
    parts = _repo_parts(REPO_URL)
    assert parts is not None, "REPO_URL should look like https://github.com/your-username/your-repo-name (Project step 1)."
    assert parts[0] != "your-username", "Paste your own repository link into REPO_URL first (Project step 1)."
    return parts

def _raw(owner, repo, path):
    r = requests.get(f"https://raw.githubusercontent.com/{owner}/{repo}/HEAD/{path}", timeout=20)
    return r.text if r.status_code == 200 else None

print("✅ GitHub checker ready")

✅ GitHub checker ready


## Project step 1: Make your public repository

**A. Make a GitHub account** (skip if you have one)

1. Go to [github.com](https://github.com) and click **Sign up**.
2. Use an email you'll keep after graduation, so the account stays yours.
3. Pick a professional **username**, like your first and last name: it is part of every link you share (`github.com/username/...`).
4. Finish the sign-up steps and confirm your email.

**B. Create the repository**

1. Signed in to GitHub, click the **+** at the top right of the page, then **New repository**.
2. **Repository name:** your app's name, short and lowercase, with hyphens instead of spaces, like `steam-front-page-picker`.
3. **Description:** paste your one sentence from your proposal.
4. Choose **Public**. (Private repos can't be graded, and your project must be public.)
5. Turn on **Add a README**.
6. Click **Create repository**. You are now on your repo's page, with one file, `README.md`. That was your **commit 1**.

**C. Paste your link below**

Copy the link from your browser's address bar (it looks like `https://github.com/username/steam-front-page-picker`) and paste it between the quotes below, replacing the example. Run the cell, then run the grader.

GitHub sometimes moves its buttons. If a name on the page is a little different, look for the closest match.

> **You should see:** your link printed back, then a ✅ from the grader

In [41]:
REPO_URL = "https://github.com/lawcarp5-bit/steam-game-value-finder"
print(REPO_URL)

https://github.com/lawcarp5-bit/steam-game-value-finder


In [42]:
# ── 🧪 GRADER: Project step 1  (DO NOT EDIT) ────────────────────────────────────────────────────────────
try:
    owner, repo = _my_repo()
    r = _gh(f"/repos/{owner}/{repo}")
    assert r.status_code != 404, f"GitHub can't find {owner}/{repo}. Check the link. If it is right, your repo may be private: on your repo page, Settings → General → Danger Zone → Change visibility → Public."
    assert r.status_code == 200, f"GitHub answered {r.status_code}. Run this grader again in a minute."
    assert r.json().get("private") is False, "Your repository is private. Make it public: Settings → General → Danger Zone → Change visibility."
    print(f"✅ Project step 1 passed: {owner}/{repo} is public")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ Couldn't check GitHub ({type(e).__name__}). Check your internet connection and run it again.")

✅ Project step 1 passed: lawcarp5-bit/steam-game-value-finder is public


## Project step 2: Put your project notebook and data in the repo

Your app lives in its own notebook, separate from this lab. You make it in Colab and save it straight into your repo.

**A. Make your project notebook**

1. In Colab: **File → New notebook in Drive**. It opens in a new tab.
2. Rename it (click the name at the top left) to your app's name with underscores, like `steam_front_page_picker.ipynb`.
3. Click **+ Text** and type your app's name and your one sentence, for example:
   `# Steam Front Page Picker` on the first line, and your sentence on the next.
4. Click **+ Code** and paste the **setup cell from the top of this lab** (the one that mounts Drive and finds `steam_games.csv`). Run it, and allow Drive access. You should see ✅ Loaded 1000 games.

**B. Save it to your repo** (this is **commit 2**)

1. In your project notebook: **File → Save a copy in GitHub**.
2. The first time, a window asks you to **sign in to GitHub and authorize Colab**. Allow it. (If nothing opens, your browser may be blocking pop-ups: allow pop-ups for colab.research.google.com and try again.)
3. In the **Repository** list, pick your repo. If it isn't listed, close the window and open it again.
4. **Branch:** keep `main`. **File path:** keep your notebook's name, like `steam_front_page_picker.ipynb`.
5. **Commit message:** `Add starter notebook`.
6. Keep **Include a link to Colab** checked, and click **OK**. Colab opens your notebook on GitHub in a new tab.

**C. Upload the data** (this is **commit 3**)

1. On your repo's page on GitHub: **Add file → Upload files**.
2. Drag in `steam_games.csv`, your project data (download it from eLC: **GitHub Project → project_data**, if it isn't on your computer).
3. Under **Commit changes**, write the message `Add Steam games data`, and click **Commit changes**.

Your repo now has 3 files: `README.md`, your notebook, and `steam_games.csv`. Honors Option students: upload your own dataset instead (if it is over 25 MB, put a link to it in your README instead).

> **You should see:** a ✅ that names the notebook it found in your repo

In [43]:
# ── 🧪 GRADER: Project step 2  (DO NOT EDIT) ────────────────────────────────────────────────────────────
try:
    owner, repo = _my_repo()
    r = _gh(f"/repos/{owner}/{repo}/contents/")
    assert r.status_code == 200, f"Couldn't list your repo's files (GitHub answered {r.status_code})."
    names = [f["name"] for f in r.json()]
    notebooks = [n for n in names if n.endswith(".ipynb")]
    assert notebooks, f"No notebook (.ipynb) in your repo yet. Your repo has: {names}. In your project notebook: File → Save a copy in GitHub (step 2B)."
    print(f"✅ Project step 2 passed: found {notebooks[0]} in your repo")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ Couldn't check GitHub ({type(e).__name__}). Check your internet connection and run it again.")

✅ Project step 2 passed: found https_github_com_lawcarp5_bit_steam_game_value_finder.ipynb in your repo


## Project step 3: Write your README draft

Your README is the front page of your repo: the first thing a visitor, a recruiter, or I read. It is written in **Markdown**: plain text where `#` makes a big heading, `##` a section heading, and `-` a bullet point. It needs five sections: **Problem**, **What it does**, **How to run it**, **Example output**, and **AI Use Disclosure**.

**A. Open the README for editing**

1. On your repo's page, click `README.md` in the file list.
2. Click the **pencil** icon at the top right of the file (**Edit this file**).
3. Delete what's there.

**B. Paste this draft and fill it in**

This draft follows the layout most GitHub projects use: a title, a one-line summary, what the project is, then sections a visitor can scan.

```
# Your App's Name

My app helps [who] decide [what] by [what it does with the data].

> A course project for **MIST 4600: Computer Programming in Business** at the University of Georgia's Terry College of Business (Fall 2026).

## Problem
Who your user is, what decision they have to make, and why it's hard without your app. (2-3 sentences)

## What it does
My app answers three questions:
1. **Sub-question 1:** ...
2. **Sub-question 2:** ...
3. **Sub-question 3:** ...

## Data
The [Steam Games Dataset](https://www.kaggle.com/datasets/artermiloff/steam-games-dataset) on Kaggle, collected in March 2025. The dataset is shared under the MIT License by its creator, artermiloff. This repo uses `steam_games.csv`, a smaller file made for the course: the 1,000 games with the most reviews.

## Built with
- Python and pandas, in Google Colab
- Coming soon: my outside tool

## How to run it
Coming soon

## Example output
Coming soon

## AI Use Disclosure
Coming soon

## Author
Your Name
```

1. Replace `Your App's Name` with your app's name, and the sentence below it with your one sentence from your proposal.
2. Keep the course line as it is.
3. Under **Problem**, write 2-3 sentences about your user and their decision.
4. Under **What it does**, put your sub-questions, one per numbered line, each with a short bold label, like `**Ratings:**`.
5. Under **Author**, put your name. (A LinkedIn link is a nice touch for recruiters, but optional.)
6. Leave `Coming soon` under the rest. You'll fill them in as your app grows.

Honors Option students: describe your own dataset under **Data**, with a link to where it comes from.

The five sections that are **required** (and checked by the grader) are **Problem**, **What it does**, **How to run it**, **Example output**, and **AI Use Disclosure**. Keep their headings spelled like that. The other sections are there to make your README look like a real project's.

**C. Check it and commit** (this is **commit 4**)

1. Click the **Preview** tab: the `#` lines should show up as headings.
2. Click **Commit changes...**, write the message `Add README draft`, and click **Commit changes** again.

Your repo's front page now shows your README below the file list.

> **You should see:** a ✅ that lists your five sections

In [44]:
# ── 🧪 GRADER: Project step 3  (DO NOT EDIT) ────────────────────────────────────────────────────────────
try:
    owner, repo = _my_repo()
    text = _raw(owner, repo, "README.md")
    assert text is not None, "Can't find README.md in your repo. Did you turn on Add a README in step 1B?"
    headings = [line.strip("# :").lower() for line in text.splitlines() if line.startswith("#")]
    needed = ["problem", "what it does", "how to run it", "example output", "ai use disclosure"]
    missing = [n for n in needed if not any(n in h for h in headings)]
    assert not missing, f"Your README is missing these section headings: {missing}. Each one goes on its own line, starting with ##."
    print("✅ Project step 3 passed: README has Problem, What it does, How to run it, Example output, and AI Use Disclosure")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ Couldn't check GitHub ({type(e).__name__}). Check your internet connection and run it again.")

✅ Project step 3 passed: README has Problem, What it does, How to run it, Example output, and AI Use Disclosure


## Project step 4: Write your first function

Pick **one of your sub-questions** from your proposal and answer it with a **function** that takes `games` as its first parameter and **returns** the answer (a number, a list, or a DataFrame). For example, for "Ratings: which games do players rate highest?":

```python
def top_rated(games, n):
    best = games.sort_values("pct_pos_total", ascending=False)
    return best[["name", "pct_pos_total"]].head(n)

print(top_rated(games, 5))
```

Write **your own** function, not this example.

**A. Open your project notebook from GitHub**

1. On your repo's page, click your notebook (`.ipynb`).
2. Click the **Open in Colab** badge at the top. (No badge? In Colab: **File → Open notebook → GitHub**, type your username, and pick your repo and notebook.)
3. Run the setup cell, so `games` is loaded.

**B. Add your function and save it** (this is **commit 5**)

1. Click **+ Code**, write your function, and below it one line that calls it and prints the result. Run the cell and check the answer makes sense.
2. **File → Save a copy in GitHub**: same repo, `main`, the **same file path** as before (so it updates your notebook, not makes a second one). Commit message: `Add` and your function's name, like `Add top_rated function`. Click **OK**.

**C. Paste it here too**

Paste your function into the cell below. Then set `my_function` to your function's name (no parentheses), and store one call of it in `my_result`.

Honors Option students: use your own dataset in your project notebook; below, you may write a function for any sub-question on `games`.

> **You should see:** your function's result printed, then a ✅ that found a function in your repo's notebook

In [62]:
# Paste your function here
def best_value(games):
    best_game = ""
    best_score = 0

    for i in range(len(games)):
        price = games["price"][i]
        hours = games["avg_playtime_hours"][i]

        if price > 0:
            score = hours / price

            if score > best_score:
                best_score = score
                best_game = games["name"][i]

    return best_game
print(best_value(games))
my_function = best_value
my_result = best_value(games)
print(my_result)

Killing Floor 2
Killing Floor 2


In [63]:
# ── 🧪 GRADER: Project step 4  (DO NOT EDIT) ────────────────────────────────────────────────────────────
try:
    assert callable(my_function), "Set my_function to your function's name, without parentheses: my_function = your_function_name"
    assert my_function.__name__ != "top_rated", "Write a function for one of your own sub-questions, not the example."
    assert my_result is not None and my_result is not ..., "my_result should hold what your function returns. Did you return (not print) the answer?"
    owner, repo = _my_repo()
    r = _gh(f"/repos/{owner}/{repo}/contents/")
    assert r.status_code == 200, f"Couldn't list your repo's files (GitHub answered {r.status_code})."
    notebooks = [f for f in r.json() if f["name"].endswith(".ipynb")]
    assert notebooks, "No notebook (.ipynb) in your repo yet. Do Project step 2 first."
    def has_def(f):
        nb = requests.get(f["download_url"], timeout=20).json()
        return any("def " in "".join(c["source"]) for c in nb.get("cells", []) if c["cell_type"] == "code")
    assert any(has_def(f) for f in notebooks), "Your repo's notebook has no function yet (no def). Add it in your project notebook, then File → Save a copy in GitHub again (step 4B)."
    print(f"✅ Project step 4 passed: {my_function.__name__} works here, and your repo's notebook has a function")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ Couldn't check GitHub ({type(e).__name__}). Check your internet connection and run it again.")

❌ Not yet: Your repo's notebook has no function yet (no def). Add it in your project notebook, then File → Save a copy in GitHub again (step 4B).


## Project step 5: Check your commit history

Each save to GitHub in the steps above made one commit. You should have **at least 5**:

| # | Step | Commit message |
|---|---|---|
| 1 | 1B: create the repo | `Initial commit` (GitHub writes it) |
| 2 | 2B: save the notebook from Colab | `Add starter notebook` |
| 3 | 2C: upload the data | `Add Steam games data` |
| 4 | 3C: edit the README | `Add README draft` |
| 5 | 4B: save your function | `Add top_rated function` (your function's name) |

To see them yourself: on your repo's page, click **Commits** (the clock icon with a number, above the file list). If you have fewer than 5, find the step you skipped above and do it.

From now on, every time you add a piece to your app, save it to GitHub with a message that starts with a verb and says what changed, like `Add price filter` or `Fix free games in top_rated`.

> **You should see:** a ✅ with your commit count, and your latest commit messages

In [ ]:
# ── 🧪 GRADER: Project step 5  (DO NOT EDIT) ────────────────────────────────────────────────────────────
try:
    owner, repo = _my_repo()
    r = _gh(f"/repos/{owner}/{repo}/commits?per_page=100")
    assert r.status_code == 200, f"Couldn't read your commit history (GitHub answered {r.status_code})."
    commits = r.json()
    assert len(commits) >= 5, f"Your repo has {len(commits)} commit(s). You need at least 5: one for each save in the steps above."
    for c in commits[:5]:
        print("  •", c["commit"]["message"].splitlines()[0])
    print(f"✅ Project step 5 passed: {len(commits)} commits")
except AssertionError as e:
    print(f"❌ Not yet: {e}")
except (KeyError, TypeError, AttributeError, ValueError) as e:
    print(f"❌ Not yet: {type(e).__name__}: {e}. Check the names and the steps in the exercise.")
except NameError as e:
    print(f"❌ Run the cells above first (Shift+Enter).  [{e}]")
except Exception as e:
    print(f"❌ Couldn't check GitHub ({type(e).__name__}). Check your internet connection and run it again.")

## Project step 6: Your outside tool

Your project needs **at least one tool from outside basic Python and pandas**: a web API like CheapShark (Week 11), an LLM API like Gemini (Week 12), or another package from PyPI. Double-click this cell and answer:

* **My outside tool:**
* **Which sub-question it helps answer:**
* **What it will do, in one sentence:** *(for example: "Gemini labels each game's description as single-player, multiplayer, or both, so I can compare their ratings.")*

Graded on being complete and specific. You can change your mind later.

---
# AI Use Disclosure (required: fill this in whether or not you used AI)

Double-click this cell and complete it. **This counts as one of the lab's 13 graded items.** Honest disclosure never costs you points; undisclosed AI use is an academic honesty violation.

* **I used AI tools while completing this work:** Yes / No
* If yes:
  * **Tools used (include all):**
  * **What I asked AI for (1-2 lines):**
  * **What I added, changed, or fixed in the AI's output (1-2 lines):**
  * **What I designed and wrote entirely myself (1-2 lines):**
* **I understand every line of code I am submitting:** Yes / No

---
## ⚠️ Last thing before you upload

1. **Runtime → Run all.** Colab will ask for Drive access again: allow it. It takes about 3 minutes, because Gemini labels 20 games. Scroll through the whole notebook: every one of the 11 grader cells must show a **✅**. An ❌ anywhere means that step is not finished. (If Gemini says 503 or 429, wait a minute and run that cell again.)
2. **Project step 6** and the **AI Use Disclosure** are filled in. Each counts as one of the 13 graded items, so don't skip them.
3. Your **repository is public**, and your link is in `REPO_URL`. I look at your repo too, so keep it public.
4. **File → Download → Download .ipynb**, and check the downloaded file is named **`Module06_FirstLast.ipynb`** with your own name, no spaces.
5. Upload it to the **Module 6 Lab Submission dropbox on eLC** by **Sunday, Nov 15, 11:59 pm**.
6. Keep this notebook in your Google Drive, with its revision history, until final grades are posted.

A submission whose graders were never re-run, or that carries the wrong file name, is the most common way to lose points that the code itself had earned.

Late policy: 10% off per day, nothing accepted after 5 days. And remember, your lowest lab is dropped. Questions → the eLC discussion forum, office hours (Tue/Thu 4:30-5:30, C425 Benson Hall), or email with "MIST 4600" in the subject.